# sklearn preprocessing & pipelines cheat sheet

**What's in here**
- Scalers: fit on train, transform test; what leakage looks like numerically
- `Pipeline` / `make_pipeline`, accessing steps, `clone`
- `ColumnTransformer`: numeric scaling + `OneHotEncoder` for categoricals
- `SimpleImputer` (with `add_indicator`), `KBinsDiscretizer`, `FunctionTransformer`, `TransformedTargetRegressor`
- `get_feature_names_out`, `set_output(transform="pandas")`
- Why pipelines make cross-validation honest
- Classification on the meters table: `LogisticRegression`, report, confusion matrix, ROC AUC, imbalance, thresholds

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
np.set_printoptions(precision=4, suppress=True)

df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"]).sort_values("time").reset_index(drop=True)
df["hour"]    = df["time"].dt.hour
df["dow"]     = df["time"].dt.dayofweek
df["month"]   = df["time"].dt.month
df["weekend"] = (df["dow"] >= 5).astype(int)
df["hdd"]     = np.clip(15 - df["temp_c"], 0, None)      # heating degrees
df["cdd"]     = np.clip(df["temp_c"] - 22, 0, None)      # cooling degrees
print(df.shape); df.head(3)

(17520, 12)


,time,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh,hour,dow,month,weekend,hdd,cdd
0,2022-01-01 00:00:00+00:00,26858.4,0.11,7.00,0.0,81.83,0,5,1,1,14.89,0.0
1,2022-01-01 01:00:00+00:00,26177.8,-0.18,6.61,0.0,88.21,1,5,1,1,15.18,0.0
2,2022-01-01 02:00:00+00:00,26229.4,-1.11,7.14,0.0,84.71,2,5,1,1,16.11,0.0


In [2]:
meters = pd.read_csv("../data/meters.csv", parse_dates=["signup_date"])
print(meters.shape)
print(meters.isna().sum().to_dict())
print("region values:", sorted(meters["region"].unique()))
meters.head(3)

(300, 7)
{'meter_id': 0, 'region': 0, 'tariff': 13, 'customer_type': 0, 'annual_kwh_estimate': 6, 'signup_date': 0, 'has_solar': 0}
region values: ['London', 'Midlands', 'North', 'Scotland', 'Wales', 'london', 'midlands', 'north', 'wales']


,meter_id,region,tariff,customer_type,annual_kwh_estimate,signup_date,has_solar
0,M100000,London,Fixed,sme,21622.0,2021-07-07,False
1,M100001,London,Fixed,residential,2286.0,2022-11-17,False
2,M100002,London,Fixed,residential,3665.0,2021-06-04,False


**Pitfall:** `region` has inconsistent casing (`London` vs `london`). `OneHotEncoder` would create separate columns for each spelling. Normalise text *before* encoding - this is a data-cleaning step, not a modelling step.

In [3]:
meters["region"] = meters["region"].str.strip().str.title()
meters["has_solar"] = meters["has_solar"].astype(int)
meters["tenure_days"] = (pd.Timestamp("2024-01-01") - meters["signup_date"]).dt.days
print("region values:", sorted(meters["region"].unique()))

region values: ['London', 'Midlands', 'North', 'Scotland', 'Wales']


## Scalers: fit on train only

`fit` learns mean/std (or min/max, or median/IQR). Fitting on the full dataset lets test-set statistics leak into training. The effect is usually small numerically - which is exactly why people get away with it - but it is wrong in principle and can matter a lot with drift.

In [4]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

split = int(len(df) * 0.8)
num_cols = ["temp_c", "wind_ms", "solar_wm2", "price_eur_mwh"]
X_tr, X_te = df[num_cols].iloc[:split], df[num_cols].iloc[split:]

sc_ok = StandardScaler().fit(X_tr)                       # correct
sc_leak = StandardScaler().fit(pd.concat([X_tr, X_te]))  # leak: test stats used

te_ok = sc_ok.transform(X_te); te_leak = sc_leak.transform(X_te)
print("train mean :", sc_ok.mean_.round(2))
print("full mean  :", sc_leak.mean_.round(2), " <- price differs most: the 2022 gas spike is in train only")
print("max abs diff in scaled test values per column:", np.abs(te_ok - te_leak).max(axis=0).round(3))

train mean : [  9.89   7.39 105.15 102.38]
full mean  : [ 9.93  7.3  97.71 98.52]  <- price differs most: the 2022 gas spike is in train only
max abs diff in scaled test values per column: [0.088 0.059 0.218 0.131]


In [5]:
pd.DataFrame({
    "StandardScaler": StandardScaler().fit(X_tr).transform(X_te)[:3, 3],
    "MinMaxScaler":   MinMaxScaler().fit(X_tr).transform(X_te)[:3, 3],
    "RobustScaler":   RobustScaler().fit(X_tr).transform(X_te)[:3, 3],
}, index=["price row0", "row1", "row2"]).round(3)

,StandardScaler,MinMaxScaler,RobustScaler
price row0,-1.264,0.168,-0.959
row1,-1.593,0.140,-1.208
row2,-1.295,0.166,-0.982


**Interview check:** "MinMaxScaler on the test set produced 1.3 - is that a bug?" No: min/max come from training; a test value above the training max maps above 1. It *is* a sign of drift worth mentioning. `RobustScaler` (median/IQR) is the choice when spikes would distort mean/std.

## Pipeline

A `Pipeline` chains transformers and a final estimator. `fit` calls `fit_transform` on each step in order using only the data it is given - so inside cross-validation each fold's scaler sees only that fold's training data. `make_pipeline` names steps automatically (lower-case class name).

In [6]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import Ridge
from sklearn.base import clone

feats = ["hour", "dow", "weekend", "temp_c", "hdd", "cdd", "wind_ms", "solar_wm2"]
y = df["consumption_mwh"]
Xtr, Xte, ytr, yte = df[feats].iloc[:split], df[feats].iloc[split:], y.iloc[:split], y.iloc[split:]

pipe = Pipeline([("scale", StandardScaler()), ("ridge", Ridge(alpha=1.0))])
pipe.fit(Xtr, ytr)
print("steps:", list(pipe.named_steps))
print("R2 test:", round(pipe.score(Xte, yte), 4))
print("coef via named_steps:", pd.Series(pipe.named_steps["ridge"].coef_, index=feats).round(0).to_dict())
print("same via index:", pipe[-1].alpha, "| via name:", pipe["ridge"].alpha)

pipe2 = make_pipeline(StandardScaler(), Ridge())
print("auto step names:", list(pipe2.named_steps))

steps: ['scale', 'ridge']
R2 test: 0.4357
coef via named_steps: {'hour': 2475.0, 'dow': 27.0, 'weekend': -1025.0, 'temp_c': 277.0, 'hdd': 2469.0, 'cdd': 15.0, 'wind_ms': 43.0, 'solar_wm2': 1295.0}
same via index: 1.0 | via name: 1.0
auto step names: ['standardscaler', 'ridge']


Parameters of nested steps use the `step__param` syntax (double underscore). `clone` gives an unfitted copy with the same parameters - use it when you want to refit the same spec on different data.

In [7]:
pipe.set_params(ridge__alpha=100)
print(pipe.get_params()["ridge__alpha"])
fresh = clone(pipe)
print("fresh is fitted?", hasattr(fresh["ridge"], "coef_"), "| original fitted?", hasattr(pipe["ridge"], "coef_"))

100
fresh is fitted? False | original fitted? True


## ColumnTransformer: different preprocessing per column type

Numeric columns get imputed + scaled; categorical columns get one-hot. `remainder="drop"` (default) silently drops columns you did not list - `remainder="passthrough"` keeps them. `handle_unknown="ignore"` avoids a crash when the test set has a category unseen in training (all-zero row instead).

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

num = ["annual_kwh_estimate", "tenure_days", "has_solar"]
cat = ["region", "tariff"]

pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore", sparse_output=False)), cat),
])
Xm = pre.fit_transform(meters[num + cat])
print("shape in:", meters[num + cat].shape, " shape out:", Xm.shape)
print(pre.get_feature_names_out())

shape in: (300, 5)  shape out: (300, 11)
['num__annual_kwh_estimate' 'num__tenure_days' 'num__has_solar'
 'cat__region_London' 'cat__region_Midlands' 'cat__region_North'
 'cat__region_Scotland' 'cat__region_Wales' 'cat__tariff_Fixed'
 'cat__tariff_TOU' 'cat__tariff_Variable']


`set_output(transform="pandas")` keeps column names through the whole pipeline - very useful for debugging *which* column went wrong.

In [9]:
pre.set_output(transform="pandas")
pre.fit_transform(meters[num + cat]).head(3).round(2)

,num__annual_kwh_estimate,num__tenure_days,num__has_solar,cat__region_London,cat__region_Midlands,cat__region_North,cat__region_Scotland,cat__region_Wales,cat__tariff_Fixed,cat__tariff_TOU,cat__tariff_Variable
0,2.22,0.74,-0.36,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,-0.46,-1.65,-0.36,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,-0.27,0.90,-0.36,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


## Imputation

`SimpleImputer` strategies: `mean`, `median`, `most_frequent`, `constant`. `add_indicator=True` appends a 0/1 "was missing" column - missingness is often informative (a missing tariff might mean a new customer). Always fit on train only; the imputed value is a learned parameter.

In [10]:
imp = SimpleImputer(strategy="median", add_indicator=True).set_output(transform="pandas")
out = imp.fit_transform(meters[["annual_kwh_estimate"]])
print("learned median:", imp.statistics_.round(0))
out[meters["annual_kwh_estimate"].isna()].head(3)

learned median: [3290.]


,annual_kwh_estimate,missingindicator_annual_kwh_estimate
43,3290.0,1.0
82,3290.0,1.0
103,3290.0,1.0


**Pitfall:** imputing a *time series* with the column median uses future values (the median is computed over the whole column). For time-ordered data use `ffill()` (last known value) - see the pandas time-series notebook.

## Binning, functions, target transforms

- `KBinsDiscretizer`: turn a continuous feature into ordinal bins or one-hot bins (e.g. temperature bands).
- `FunctionTransformer`: wrap any numpy function (log1p for skewed features).
- `TransformedTargetRegressor`: fit on a transformed *target* (log) and automatically invert predictions.

In [11]:
from sklearn.preprocessing import KBinsDiscretizer, FunctionTransformer
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression

kb = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile")
bands = kb.fit_transform(Xtr[["temp_c"]])
print("temp bin edges:", kb.bin_edges_[0].round(1))

log_tf = FunctionTransformer(np.log1p, inverse_func=np.expm1, validate=True)
print("log1p of annual kWh:", log_tf.transform(meters[["annual_kwh_estimate"]].fillna(0).values[:3]).ravel().round(2))

ttr = TransformedTargetRegressor(regressor=make_pipeline(StandardScaler(), Ridge()), func=np.log, inverse_func=np.exp)
ttr.fit(Xtr, ytr)
print("predictions are back in MWh:", ttr.predict(Xte[:3]).round(0))

temp bin edges: [-6.4  3.   7.4 12.6 16.6 27.7]
log1p of annual kWh: [9.98 7.73 8.21]
predictions are back in MWh: [22879. 23172. 23459.]


## Why pipelines make cross-validation honest

Without a pipeline, people scale the full X once and then cross-validate - every fold has seen the test fold's statistics. With the pipeline, `cross_val_score` refits the scaler inside each fold.

In [12]:
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)

X_all = df[feats]
# honest: scaler inside the pipeline
s_ok = cross_val_score(make_pipeline(StandardScaler(), Ridge(alpha=1)), X_all, y, cv=tscv, scoring="neg_root_mean_squared_error")
# leaky: scaler fitted once on everything
X_scaled_all = StandardScaler().fit_transform(X_all)
s_leak = cross_val_score(Ridge(alpha=1), X_scaled_all, y, cv=tscv, scoring="neg_root_mean_squared_error")
print("RMSE per fold, pipeline:", (-s_ok).round(0))
print("RMSE per fold, leaky   :", (-s_leak).round(0))
print("(tiny difference for Ridge on scaled inputs - the principle matters more with imputers, target encoders, PCA, feature selection)")

RMSE per fold, pipeline: [3121. 3025. 2884. 2841. 3002.]
RMSE per fold, leaky   : [3095. 3026. 2884. 2841. 3002.]
(tiny difference for Ridge on scaled inputs - the principle matters more with imputers, target encoders, PCA, feature selection)


## Classification on the meters table

Predict `customer_type` (residential / sme) from consumption estimate, tenure, region, tariff. Full pipeline: `ColumnTransformer` -> `LogisticRegression`. Stratified split keeps class proportions (fine here: rows are independent customers, not a time series).

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

Xc = meters[num + cat]; yc = (meters["customer_type"] == "sme").astype(int)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3, stratify=yc, random_state=0)
print("class balance train:", yc_tr.mean().round(3), " test:", yc_te.mean().round(3))

clf = make_pipeline(clone(pre), LogisticRegression(max_iter=1000))
clf.fit(Xc_tr, yc_tr)
proba = clf.predict_proba(Xc_te)[:, 1]
pred = clf.predict(Xc_te)                      # = proba >= 0.5
print("accuracy:", round(accuracy_score(yc_te, pred), 3), " ROC AUC:", round(roc_auc_score(yc_te, proba), 3))
print(confusion_matrix(yc_te, pred))
print(classification_report(yc_te, pred, target_names=["residential", "sme"]))

class balance train: 0.114  test: 0.111


accuracy: 1.0  ROC AUC: 1.0
[[80  0]
 [ 0 10]]
              precision    recall  f1-score   support

 residential       1.00      1.00      1.00        80
         sme       1.00      1.00      1.00        10

    accuracy                           1.00        90
   macro avg       1.00      1.00      1.00        90
weighted avg       1.00      1.00      1.00        90



Perfect score. **Interview check:** "Accuracy 1.0 - what do you do?" Get suspicious first: is a feature a proxy for the label? Here it is by construction - SMEs use ~8x the energy of households, so `annual_kwh_estimate` separates the classes completely. In real data a perfect classifier usually means the label leaked into a feature (e.g. a tariff only offered to businesses).

Coefficients of a logistic regression are log-odds per (standardised) unit. Read them through `get_feature_names_out` of the preprocessing step.

In [14]:
names = clf[0].get_feature_names_out()
pd.Series(clf[-1].coef_[0], index=names).sort_values().round(2)

num__has_solar             -0.30
cat__region_Scotland       -0.23
cat__region_North          -0.20
cat__tariff_TOU            -0.16
num__tenure_days           -0.14
cat__tariff_Fixed          -0.05
cat__region_London          0.05
cat__region_Wales           0.11
cat__tariff_Variable        0.22
cat__region_Midlands        0.26
num__annual_kwh_estimate    3.13
dtype: float64

**Pitfall - accuracy on imbalanced classes.** Predicting `has_solar` (12% positive): a model that always says "no" is 88% accurate. Compare with `DummyClassifier`, look at recall/precision and ROC AUC, and consider `class_weight="balanced"`.

In [15]:
from sklearn.dummy import DummyClassifier
ys = meters["has_solar"]; Xs = meters[["annual_kwh_estimate", "tenure_days", "region", "tariff"]]
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(Xs, ys, test_size=0.3, stratify=ys, random_state=0)
pre_s = ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), ["annual_kwh_estimate", "tenure_days"]),
                           ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), ["region", "tariff"])])
for name, model in [("dummy most_frequent", DummyClassifier(strategy="most_frequent")),
                    ("logreg", make_pipeline(clone(pre_s), LogisticRegression(max_iter=1000))),
                    ("logreg balanced", make_pipeline(clone(pre_s), LogisticRegression(max_iter=1000, class_weight="balanced")))]:
    model.fit(Xs_tr, ys_tr)
    p = model.predict(Xs_te)
    pr = model.predict_proba(Xs_te)[:, 1] if hasattr(model, "predict_proba") else p
    tn, fp, fn, tp = confusion_matrix(ys_te, p).ravel()
    print(f"{name:20s} acc={accuracy_score(ys_te, p):.3f}  auc={roc_auc_score(ys_te, pr):.3f}  tp={tp} fp={fp} fn={fn} tn={tn}")

dummy most_frequent  acc=0.889  auc=0.500  tp=0 fp=0 fn=10 tn=80


logreg               acc=0.889  auc=0.481  tp=0 fp=0 fn=10 tn=80
logreg balanced      acc=0.567  auc=0.496  tp=5 fp=34 fn=5 tn=46


There is no real signal for `has_solar` in these columns, and the AUC near 0.5 says so honestly while the accuracy of the dummy looks fine. **Interview check:** "88% accuracy - good model?" No: same as always predicting the majority class.

## A realistic imbalanced problem: price-spike hours

Classify whether the hour's price is in the top 10% of the training distribution, from things known at the time (consumption, wind, solar, hour, last hour's price). Chronological split, because this is a time series. Here precision/recall trade off for real.

In [16]:
from sklearn.metrics import precision_score, recall_score, average_precision_score
sp = df.copy()
sp["price_lag1"] = sp["price_eur_mwh"].shift(1)
sp = sp.dropna().reset_index(drop=True)
sp_split = int(len(sp) * 0.8)
thr_spike = sp["price_eur_mwh"].iloc[:sp_split].quantile(0.9)          # threshold from TRAIN only
sp["spike"] = (sp["price_eur_mwh"] > thr_spike).astype(int)
sf = ["consumption_mwh", "wind_ms", "solar_wm2", "hour", "weekend", "price_lag1"]
Xs_tr, Xs_te = sp[sf].iloc[:sp_split], sp[sf].iloc[sp_split:]
ys_tr, ys_te = sp["spike"].iloc[:sp_split], sp["spike"].iloc[sp_split:]
print(f"spike threshold {thr_spike:.1f} EUR/MWh | positive rate train {ys_tr.mean():.3f}, test {ys_te.mean():.3f}")

spike_clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xs_tr, ys_tr)
proba_s = spike_clf.predict_proba(Xs_te)[:, 1]
print("ROC AUC:", round(roc_auc_score(ys_te, proba_s), 3), " average precision:", round(average_precision_score(ys_te, proba_s), 3),
      " (a random model has AP =", round(ys_te.mean(), 3), ")")
print(confusion_matrix(ys_te, spike_clf.predict(Xs_te)))
print(classification_report(ys_te, spike_clf.predict(Xs_te), target_names=["normal", "spike"]))

spike threshold 148.3 EUR/MWh | positive rate train 0.100, test 0.016
ROC AUC: 0.839  average precision: 0.202  (a random model has AP = 0.016 )
[[3389   60]
 [  31   24]]
              precision    recall  f1-score   support

      normal       0.99      0.98      0.99      3449
       spike       0.29      0.44      0.35        55

    accuracy                           0.97      3504
   macro avg       0.64      0.71      0.67      3504
weighted avg       0.98      0.97      0.98      3504



Note the positive rate is lower in the test period than in training (prices fell after the 2022 gas spike): the threshold was fixed on training data, as it must be, but the class balance drifted. Recall on spikes is poor at the default 0.5 threshold.

## Thresholds

`predict` uses 0.5. In practice you pick the threshold from the cost of false positives vs false negatives (missing a spike costs more than a false alarm if you are hedging). Sweep it on `predict_proba`.

In [17]:
rows = []
for thr in [0.05, 0.1, 0.2, 0.3, 0.5, 0.7]:
    p = (proba_s >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(ys_te, p).ravel()
    rows.append({"threshold": thr, "precision": precision_score(ys_te, p, zero_division=0), "recall": recall_score(ys_te, p),
                 "flagged_hours": int(p.sum()), "fp": fp, "fn": fn})
pd.DataFrame(rows).round(3)

,threshold,precision,recall,flagged_hours,fp,fn
0,0.05,0.055,0.727,728,688,15
1,0.10,0.083,0.727,484,444,15
2,0.20,0.138,0.691,276,238,17
3,0.30,0.170,0.564,182,151,24
4,0.50,0.286,0.436,84,60,31
5,0.70,0.366,0.273,41,26,40


**Interview check:** "Which threshold?" Depends on the cost asymmetry; there is no statistical answer. Report the whole curve (or AUC / average precision) and let the business constraint pick the operating point. Also: `class_weight="balanced"` moves the default operating point but does not add information - it is equivalent to changing the threshold for logistic regression.

## Persisting a fitted pipeline

`joblib.dump(pipe, "model.joblib")` / `joblib.load(...)`. Store the *whole pipeline* (preprocessing + model), never the model alone, otherwise production data gets scaled differently from training data. Pin the sklearn version - pickles are not portable across versions. (Not executed here to avoid writing files.)

## Quick reference

| Task | Call |
|---|---|
| Scale (train only) | `sc = StandardScaler().fit(X_tr); sc.transform(X_te)` |
| Chain | `make_pipeline(StandardScaler(), Ridge())` |
| Per-column | `ColumnTransformer([("num", ..., num_cols), ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])` |
| Impute | `SimpleImputer(strategy="median", add_indicator=True)` |
| Names out | `pipe[:-1].get_feature_names_out()` / `.set_output(transform="pandas")` |
| Step params | `pipe.set_params(ridge__alpha=10)`; `pipe.named_steps["ridge"].coef_` |
| Log target | `TransformedTargetRegressor(regressor=..., func=np.log, inverse_func=np.exp)` |
| Classification report | `classification_report(y, pred)`, `roc_auc_score(y, proba)` |